# Tutorial 2: Audio Generation with Room Accoustic

<p align="right" style="margin-right: 8px;">
    <a target="_blank" href="https://colab.research.google.com/github/idiap/sdialog/blob/main/tutorials/01_audio/2.accoustic_simulation.ipynb">
        <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
    </a>
</p>

## Getting started

### Environment Setup

Let's first check if our environment is all set up.

In [ ]:
# Setup the environment depending on weather we are running in Google Colab or Jupyter Notebook
import os
from IPython import get_ipython
from IPython.display import Audio, display

if "google.colab" in str(get_ipython()):
    print("Running on CoLab")

    !sudo apt-get install sox ffmpeg
    !sudo apt-get -qq -y install espeak-ng > /dev/null 2>&1
    %pip install -q kokoro>=0.9.4
    # Installing sdialog
    !git clone https://github.com/idiap/sdialog.git
    %cd sdialog
    %pip install -e .[audio]
    %cd ..
else:
    print("Running in Jupyter Notebook")

### Load the example dialogue

In order to run the next steps in a fast manner, we will start from an existing dialog generated using previous tutorials:

In [ ]:
from sdialog import Dialog

path_dialog = "../../tests/data/demo_dialog_doctor_patient.json"

if not os.path.exists(path_dialog):
    !wget https://raw.githubusercontent.com/idiap/sdialog/refs/heads/main/tests/data/demo_dialog_doctor_patient.json
    path_dialog = "./demo_dialog_doctor_patient.json"

original_dialog = Dialog.from_file(path_dialog)
original_dialog.print()

# Tutorial

### Instanciate voices database

In [ ]:
from sdialog.audio.tts import KokoroTTS
from sdialog.audio.voice_database import HuggingfaceVoiceDatabase

tts_engine = KokoroTTS()
kokoro_voice_database = HuggingfaceVoiceDatabase("sdialog/voices-kokoro")

## Setup stage: Audio Dialog and Audio Pipeline

In [ ]:
from sdialog.audio.dialog import AudioDialog
dialog: AudioDialog = AudioDialog.from_dialog(original_dialog)

In [ ]:
from sdialog.audio.pipeline import AudioPipeline
os.makedirs("./outputs_evaluation", exist_ok=True)
audio_pipeline = AudioPipeline(
    voice_database=kokoro_voice_database,
    tts_engine=tts_engine,
    dir_audio="./outputs_evaluation",
)

In [ ]:
from sdialog.audio.utils import SourceVolume, SourceType
from sdialog.audio.room import SpeakerSide, Role, RoomPosition
from sdialog.audio.jsalt import MedicalRoomGenerator, RoomRole

In [ ]:
room = MedicalRoomGenerator().generate(args={"room_type": RoomRole.EXAMINATION})
room.place_speaker_around_furniture(speaker_name=Role.SPEAKER_1, furniture_name="desk", max_distance=1.0, side=SpeakerSide.FRONT)
room.place_speaker_around_furniture(speaker_name=Role.SPEAKER_2, furniture_name="desk", max_distance=1.0, side=SpeakerSide.BACK)

In [ ]:
new_audio_dialog = original_dialog.to_audio(
    tts_engine=tts_engine,
    perform_room_acoustics=True,
    room=room,
    dialog_dir_name="evaluation_demo",
    room_name="evaluation_demo_1",
    background_effect="white_noise",
    foreground_effect="ac_noise_minimal",
    foreground_effect_position=RoomPosition.TOP_RIGHT,
    source_volumes={
        SourceType.ROOM: SourceVolume.HIGH,
        SourceType.BACKGROUND: SourceVolume.VERY_LOW
    },
    kwargs_pyroom={
        "ray_tracing": True,
        "air_absorption": True
    },
    override_tts_audio=True
)

In [ ]:
new_audio_dialog.display()

# Evaluating audios

### Speaker consistency

In [ ]:
new_audio_dialog.speakers_names

In [ ]:
current_room = new_audio_dialog.audio_step_3_filepaths["evaluation_demo_1"]["room"]
print("Speakers distances to the microphone (2D):", current_room.get_speaker_distances_to_microphone(2))
print("Speakers distances to the microphone (3D):", current_room.get_speaker_distances_to_microphone(3))

In [ ]:
from sdialog.audio.evaluation import evaluate, SpeakerConsistency

res = evaluate([new_audio_dialog], [
    SpeakerConsistency(use_acoustic_audio=False)
])

In [ ]:
print(res)